# 03 · Salary model

**Question:** how well can we predict a posting's annual pay from what the posting says (title, employer, place, time), and what drives pay?

**Data:** jobs with an extracted salary in 10k–500k USD. The target is `log_pay` = log(annual midpoint).

**Honest evaluation:**
- **Time holdout:** fit on postings created before 2025-10-01, calibrate the intervals on 2025-10 → 2025-12, and **test on 2026**.
- **Unseen-employer stress test:** hold out 20% of *companies* entirely.
- Encodings of company and title are computed **out-of-fold** (grouped by `BASE_HASH`), so no posting or repost sees its own pay.

**Features are only what an undisclosed posting also has.** That's why `pay_frequency` / `has_range` (derived from the salary text) are **excluded**. The model can then be applied to the ~70% of postings with no pay listed.

**Models:** global median → title-cluster median → Ridge → **LightGBM**, then LightGBM quantile models + **conformalized quantile regression (CQR)** for 90% prediction intervals.

**Outputs:** `salary_predictions.parquet` (a prediction + interval for *every* job), `salary_model_metrics.json`, and figures.

In [ ]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from src import features as F

cfg = F.get_config()
SAMPLE_FRAC = cfg['SAMPLE_FRAC']      # set < 1.0 for a quick trial run (e.g. 0.05)
FORCE_REBUILD = cfg['FORCE_REBUILD']  # True = ignore cached outputs and recompute
SUFFIX = F.sample_suffix(cfg)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 180)
print('root:', ROOT, '| SAMPLE_FRAC:', SAMPLE_FRAC, '| FORCE_REBUILD:', FORCE_REBUILD)

In [ ]:
import json
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from src import modeling as M

CAL_START, TEST_START = '2025-10-01', '2026-01-01'
ALPHA = 0.10            # 90% prediction intervals
N_CPU = F.cpu_limit()
TUNE = True             # small num_leaves grid; set False to skip
print('usable CPUs:', N_CPU)

## 1. Load and join

In [ ]:
jobs = pd.read_parquet(cfg['DATA_PROC'] / f'jobs_model{SUFFIX}.parquet')
tf = pd.read_parquet(cfg['DATA_PROC'] / f'title_features{SUFFIX}.parquet')
pca_cols = [c for c in tf.columns if c.startswith('title_pca_')]
df = jobs.merge(tf[['title_clean', 'core_title', 'title_cluster', 'cluster_label'] + pca_cols], on='title_clean', how='left')
assert df['title_cluster'].notna().all()
df['zip3'] = df['zip5'].astype('string').str[:3]
df['days_since_start'] = (df['CREATED'] - df['CREATED'].min()).dt.days.astype('int32')
df['split'] = np.where(df['CREATED'] >= TEST_START, 'test', np.where(df['CREATED'] >= CAL_START, 'cal', 'train'))
lab = df[df['log_pay'].notna()].reset_index(drop=True)
print(f"{len(df):,} jobs; {len(lab):,} with pay in range")
lab['split'].value_counts().reindex(['train', 'cal', 'test'])

## 2. Features
- **Title semantics:** 32 PCA dims of the embedding + `title_cluster` (categorical).
- **Title flags:** seniority, level, part-time, bonus, etc.
- **Target encodings (out-of-fold):** company, core title, company × cluster. Company alone doesn't say *what* job it is, so company × cluster captures "nurse at HCA".
  - Employers and titles with no salaried training postings get **NaN**, not the global average, and LightGBM routes them to a learned fallback. Filling them with the average would inflate predictions for low-wage jobs at employers that never disclose pay.
- **Hiring context:** ATS domain, city, zip3 (top levels, rest → other), company size, repost count.
- **Time:** month, weekday, days since start (trend).

In [ ]:
flag_cols = [c for c in jobs.columns if c.startswith('title_') and c not in ('title_clean',)]
num_cols = pca_cols + flag_cols + ['log_company_n_posts', 'is_repost', 'base_n_posts', 'created_moy', 'created_dow', 'days_since_start']
te_specs = {'te_company': ['COMPANY_ID'], 'te_core_title': ['core_title'], 'te_company_cluster': ['COMPANY_ID', 'title_cluster']}
cat_specs = {'title_cluster': 150, 'ats_domain': 100, 'city_clean': 100, 'zip3': 60}

def te_key(frame, cols):
    return frame[cols].astype('string').fillna('na').agg('|'.join, axis=1)

def build_features(fit_rows, apply_frames):
    """Fit encoders on fit_rows (OOF for them) and apply to each frame in apply_frames."""
    X_fit = fit_rows[num_cols].copy()
    X_apply = [f[num_cols].copy() for f in apply_frames]
    encoders = {}
    for name, cols in te_specs.items():
        enc = M.TargetEncoder(smoothing=20)
        X_fit[name] = enc.fit_oof(te_key(fit_rows, cols), fit_rows['log_pay'], fit_rows['BASE_HASH'])
        for X, f in zip(X_apply, apply_frames):
            X[name] = enc.transform(te_key(f, cols))
        encoders[name] = enc
    for col, k in cat_specs.items():
        fit_cat, others = M.cap_categories(fit_rows[col], [f[col] for f in apply_frames], top_k=k)
        X_fit[col] = fit_cat
        for X, c in zip(X_apply, others):
            X[col] = c
    return X_fit, X_apply, encoders

train, cal, test = (lab[lab['split'] == s].reset_index(drop=True) for s in ['train', 'cal', 'test'])
X_train, (X_cal, X_test), encoders = build_features(train, [cal, test])
y_train, y_cal, y_test = train['log_pay'].to_numpy(), cal['log_pay'].to_numpy(), test['log_pay'].to_numpy()
cat_cols = list(cat_specs)
print('features:', X_train.shape[1], '| train/cal/test:', len(X_train), len(X_cal), len(X_test))
print('share of rows with unseen key (NaN encoding):')
display(pd.DataFrame({n: [X[te].isna().mean() for te in te_specs] for n, X in
                      [('train (OOF)', X_train), ('cal', X_cal), ('test', X_test)]}, index=list(te_specs)).round(3))

## 3. Baselines and Ridge

In [ ]:
results = {}
results['Global median'] = M.regression_metrics(y_test, np.full(len(y_test), np.median(y_train)))
cluster_med = train.groupby('title_cluster')['log_pay'].median()
results['Title-cluster median'] = M.regression_metrics(
    y_test, test['title_cluster'].map(cluster_med).fillna(np.median(y_train)).to_numpy())

ridge_num = [c for c in X_train.columns if c not in cat_cols]
ridge = make_pipeline(
    ColumnTransformer([('num', make_pipeline(SimpleImputer(strategy='mean', add_indicator=True), StandardScaler()), ridge_num),
                       ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat_cols)]),
    Ridge(alpha=10.0))
ridge.fit(X_train, y_train)
results['Ridge'] = M.regression_metrics(y_test, ridge.predict(X_test))
pd.DataFrame(results).T

## 4. LightGBM (point model)
Early stopping uses a held-out 10% of training `BASE_HASH` groups. The small grid over `num_leaves` is scored on the same holdout, never on test.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=cfg['RANDOM_STATE'])
fit_idx, es_idx = next(gss.split(X_train, y_train, train['BASE_HASH']))

BASE_PARAMS = dict(learning_rate=0.05, min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8,
                   bagging_freq=1, lambda_l2=1.0, num_threads=N_CPU, verbose=-1, seed=cfg['RANDOM_STATE'],
                   max_cat_to_onehot=8, cat_smooth=20)

def train_lgb(params, X, y, idx_fit, idx_es, rounds=5000):
    dfit = lgb.Dataset(X.iloc[idx_fit], y[idx_fit], categorical_feature=cat_cols, free_raw_data=False)
    des = lgb.Dataset(X.iloc[idx_es], y[idx_es], categorical_feature=cat_cols, reference=dfit)
    return lgb.train(params, dfit, rounds, valid_sets=[des],
                     callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])

grid = [31, 127, 255] if TUNE else [127]
tuning = []
for leaves in grid:
    t0 = time.time()
    m = train_lgb({**BASE_PARAMS, 'objective': 'regression', 'metric': 'l1', 'num_leaves': leaves},
                  X_train, y_train, fit_idx, es_idx)
    tuning.append({'num_leaves': leaves, 'best_iter': m.best_iteration,
                   'holdout_L1_log': m.best_score['valid_0']['l1'], 'seconds': round(time.time() - t0)})
tuning = pd.DataFrame(tuning)
best_leaves = int(tuning.loc[tuning['holdout_L1_log'].idxmin(), 'num_leaves'])
display(tuning)
print('best num_leaves:', best_leaves)

In [ ]:
PARAMS = {**BASE_PARAMS, 'num_leaves': best_leaves}
model = train_lgb({**PARAMS, 'objective': 'regression', 'metric': 'l1'}, X_train, y_train, fit_idx, es_idx)
pred_test = model.predict(X_test, num_iteration=model.best_iteration)
results['LightGBM'] = M.regression_metrics(y_test, pred_test)
res_table = pd.DataFrame(results).T
res_table.style.format({'MAE_$': '{:,.0f}', 'MedAE_$': '{:,.0f}', 'MAPE_%': '{:.1f}', 'R2_log': '{:.3f}', 'n': '{:,.0f}'})

## 5. Stress test: employers the model has never seen
Hold out 20% of companies from the training period. Their company encodings fall back to the prior, so the model must rely on title, place and time.

In [ ]:
gss_c = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=cfg['RANDOM_STATE'])
seen_idx, unseen_idx = next(gss_c.split(train, groups=train['COMPANY_ID'].astype(str)))
tr_s, te_u = train.iloc[seen_idx].reset_index(drop=True), train.iloc[unseen_idx].reset_index(drop=True)
Xs, (Xu,), _ = build_features(tr_s, [te_u])
f_i, e_i = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=1).split(Xs, groups=tr_s['BASE_HASH']))
m_u = train_lgb({**PARAMS, 'objective': 'regression', 'metric': 'l1'}, Xs, tr_s['log_pay'].to_numpy(), f_i, e_i)
stress = M.regression_metrics(te_u['log_pay'], m_u.predict(Xu, num_iteration=m_u.best_iteration))
cm = tr_s.groupby('title_cluster')['log_pay'].median()
stress_base = M.regression_metrics(te_u['log_pay'], te_u['title_cluster'].map(cm).fillna(tr_s['log_pay'].median()))
display(pd.DataFrame({'LightGBM, unseen companies': stress, 'Title-cluster median, unseen companies': stress_base}).T)

# Calibration for unseen employers, grouped by *predicted* pay: "when the model says $X for an employer it has
# never seen, what is the truth on average?" This is the question that matters for never-disclosing employers.
# (Grouping by true pay would show spurious bias at the extremes: regression to the mean.)
u = te_u[['log_pay']].assign(pred=m_u.predict(Xu, num_iteration=m_u.best_iteration))
u['pred_quintile'] = pd.qcut(np.exp(u['pred']), 5, labels=['Q1 lowest', 'Q2', 'Q3', 'Q4', 'Q5 highest'])
stress_bias = u.groupby('pred_quintile', observed=True).apply(
    lambda g: pd.Series({'n': len(g), 'median_predicted_$': np.exp(g['pred'].median()),
                         'median_actual_$': np.exp(g['log_pay'].median()),
                         'bias_%': (np.exp((g['pred'] - g['log_pay']).mean()) - 1) * 100,
                         'MedAPE_%': np.median(np.abs(np.exp(g['pred'] - g['log_pay']) - 1)) * 100}),
    include_groups=False)
print('Unseen employers: calibration by predicted-pay quintile (bias + = over-predicts)')
stress_bias.round(1)

## 6. Prediction intervals: conformalized quantile regression
1. Fit LightGBM quantile models at 5% and 95%.
2. On the **calibration** months (2025-Q4), compute how far outside the band the true pay falls.
3. Widen the band by the (1−α) quantile of those scores.
4. Check coverage on 2026.

In [ ]:
q_models = {}
for q in [ALPHA / 2, 1 - ALPHA / 2]:
    q_models[q] = train_lgb({**PARAMS, 'objective': 'quantile', 'alpha': q, 'metric': 'quantile'},
                            X_train, y_train, fit_idx, es_idx)
def raw_band(X):
    return (q_models[ALPHA / 2].predict(X, num_iteration=q_models[ALPHA / 2].best_iteration),
            q_models[1 - ALPHA / 2].predict(X, num_iteration=q_models[1 - ALPHA / 2].best_iteration))

lo_c, hi_c = raw_band(X_cal)
qhat = M.cqr_qhat(y_cal, lo_c, hi_c, ALPHA)
lo_t, hi_t = raw_band(X_test)
interval_table = pd.DataFrame({
    'raw quantile band': M.interval_metrics(y_test, lo_t, hi_t),
    'conformalized (CQR)': M.interval_metrics(y_test, lo_t - qhat, hi_t + qhat),
}).T
print(f'target coverage {1-ALPHA:.0%}; CQR adjustment on log scale: {qhat:+.3f}')
interval_table

In [ ]:
# Coverage by job family: is the interval honest for every kind of job, not just on average?
cov = pd.DataFrame({'cluster_label': test['cluster_label'], 'y': y_test, 'lo': lo_t - qhat, 'hi': hi_t + qhat})
cov['covered'] = (cov['y'] >= cov['lo']) & (cov['y'] <= cov['hi'])
cov['rel_width'] = np.exp(cov['hi']) / np.exp(cov['lo'])
by_cl = cov.groupby('cluster_label').agg(n=('covered', 'size'), coverage=('covered', 'mean'), width_ratio=('rel_width', 'median'))
by_cl = by_cl[by_cl['n'] >= 30].sort_values('coverage')
print(f'coverage across {len(by_cl)} clusters with n>=30: min {by_cl.coverage.min():.2f}, median {by_cl.coverage.median():.2f}')
by_cl.head(10)

## 7. What drives pay? (SHAP)
The contributions come from LightGBM's native TreeSHAP (`pred_contrib`). The 32 title-embedding dims are summed into one "title meaning" feature, so the chart reads naturally.

In [ ]:
rng = np.random.default_rng(cfg['RANDOM_STATE'])
sidx = rng.choice(len(X_test), size=min(20_000, len(X_test)), replace=False)
Xs_shap = X_test.iloc[sidx]
contrib = model.predict(Xs_shap, num_iteration=model.best_iteration, pred_contrib=True)
shap_df = pd.DataFrame(contrib[:, :-1], columns=X_test.columns)
shap_grouped = shap_df.drop(columns=pca_cols).assign(**{'title meaning (embedding)': shap_df[pca_cols].sum(axis=1)})
importance = shap_grouped.abs().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
importance.head(20).iloc[::-1].plot.barh(ax=ax)
ax.set(title='What drives predicted pay (mean |SHAP|, log-pay units)', xlabel='mean |SHAP value|  (0.1 ≈ 10% pay)')
plt.tight_layout(); fig.savefig(cfg['FIG_DIR'] / f'03_shap_importance{SUFFIX}.png', dpi=150); plt.show()

In [ ]:
# Beeswarm for the non-embedding features (direction of effects)
try:
    import shap
    plot_cols = [c for c in importance.index[:16] if c in X_test.columns and c not in cat_cols]
    shap.summary_plot(shap_df[plot_cols].to_numpy(), Xs_shap[plot_cols], show=False, max_display=15)
    plt.title('Direction of effects (numeric features)')
    plt.tight_layout(); plt.savefig(cfg['FIG_DIR'] / f'03_shap_beeswarm{SUFFIX}.png', dpi=150, bbox_inches='tight'); plt.show()
except ImportError:
    print('shap not installed - skipping beeswarm')

In [ ]:
# Case studies: four test postings, what the model predicted and why
def explain(i, k=6):
    row = test.iloc[sidx[i]]
    contrib_i = shap_grouped.iloc[i].sort_values(key=np.abs, ascending=False).head(k)
    base = contrib[i, -1]
    pred = base + shap_df.iloc[i].sum()
    print(f"{row['TITLE']!s} @ {row['COMPANY_NAME']} ({row['city_clean']})")
    print(f"   actual ${np.exp(row['log_pay']):,.0f} | predicted ${np.exp(pred):,.0f} | "
          f"90% interval ${np.exp(lo_t[sidx[i]] - qhat):,.0f}-${np.exp(hi_t[sidx[i]] + qhat):,.0f}")
    print('   ' + ', '.join(f'{n}: {v:+.2f}' for n, v in contrib_i.items()) + '\n')

for i in rng.choice(len(sidx), size=4, replace=False):
    explain(i)

## 8. Predict pay for every posting
The final models were fit on the training period. The same encoders are applied to **all** jobs, including the ~70% that don't list pay.

**Caveat for the undisclosed group:** the model learned only from postings that *do* disclose.
- Many hidden-pay postings come from employers that **never** disclose. For those, the model extrapolates from title, employer size, ATS and place, and cannot know the employer's own pay level.
- The unseen-employer stress test (section 5) measures how biased that extrapolation is, including by pay level. Quote that bias alongside any claim about hidden pay.
- Present these numbers as **"what comparable postings that list pay offer"**, not as what hidden-pay employers actually pay.

In [ ]:
all_X = build_features(train, [df])[1][0]
pred_all = model.predict(all_X, num_iteration=model.best_iteration)
lo_all, hi_all = raw_band(all_X)
preds = pd.DataFrame({
    'JOB_HASH': df['JOB_HASH'], 'has_salary': df['has_salary'], 'split': df['split'],
    'pred_log_pay': pred_all.astype('float32'),
    'pred_pay': np.exp(pred_all).round(0).astype('float32'),
    'pred_lo': np.exp(lo_all - qhat).round(0).astype('float32'),
    'pred_hi': np.exp(hi_all + qhat).round(0).astype('float32'),
    'observed_pay': df['annual_mid'].where(df['pay_in_range']).astype('float32'),
})
preds.to_parquet(cfg['DATA_PROC'] / f'salary_predictions{SUFFIX}.parquet', index=False)
print(f"saved predictions for {len(preds):,} jobs ({(preds.has_salary == 0).sum():,} without listed pay)")
preds.groupby('has_salary')[['pred_pay', 'observed_pay']].median()

In [ ]:
# Within the same job family: do postings that hide pay look like better- or worse-paid jobs?
cmp = df[['cluster_label', 'has_salary']].assign(pred_pay=preds['pred_pay'].to_numpy())
top_cl = cmp['cluster_label'].value_counts().head(15).index
g = (cmp[cmp['cluster_label'].isin(top_cl)].groupby(['cluster_label', 'has_salary'])['pred_pay'].median()
        .unstack().rename(columns={0: 'hidden pay', 1: 'listed pay'}))
g['hidden vs listed'] = g['hidden pay'] / g['listed pay'] - 1
g = g.sort_values('hidden vs listed')

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(g.index.str.slice(0, 50), g['hidden vs listed'] * 100)
ax.axvline(0, color='k', lw=0.8)
ax.set(title='Model-estimated pay: postings that hide pay vs. those that list it\n(same job family, top 15; hidden-pay estimates are extrapolations)',
       xlabel='% difference in median predicted pay')
plt.tight_layout(); fig.savefig(cfg['FIG_DIR'] / f'03_hidden_vs_listed{SUFFIX}.png', dpi=150); plt.show()
g.style.format({'hidden pay': '${:,.0f}', 'listed pay': '${:,.0f}', 'hidden vs listed': '{:+.1%}'})

## 9. Save metrics (model card)

In [ ]:
card = {
    'data': {'train': [str(train.CREATED.min().date()), CAL_START], 'calibration': [CAL_START, TEST_START],
             'test': [TEST_START, str(test.CREATED.max().date())],
             'n_train': len(train), 'n_cal': len(cal), 'n_test': len(test)},
    'target': 'log(annual pay midpoint, USD), pay in 10k-500k',
    'excluded_features': ['pay_frequency', 'has_range', 'range_width_pct (derived from the salary text)'],
    'lightgbm': {'num_leaves': best_leaves, 'best_iteration': model.best_iteration},
    'test_metrics': res_table.to_dict(orient='index'),
    'unseen_company_stress_test': {'lightgbm': stress, 'cluster_median': stress_base,
                                   'bias_by_pay_quintile': stress_bias.round(2).to_dict(orient='index')},
    'intervals': {'alpha': ALPHA, 'cqr_qhat_log': qhat, 'test': interval_table.to_dict(orient='index')},
    'top_drivers': importance.head(10).round(4).to_dict(),
}
with open(cfg['DATA_PROC'] / f'salary_model_metrics{SUFFIX}.json', 'w') as f:
    json.dump(card, f, indent=2, default=float)
model.save_model(str(cfg['DATA_PROC'] / f'salary_lgbm{SUFFIX}.txt'), num_iteration=model.best_iteration)
print(json.dumps(card['test_metrics']['LightGBM'], indent=1, default=float))